# 第三章：条件分支与循环

## 学习目标
- 用 `add_conditional_edges` 实现动态路由
- 理解路由函数的写法和返回值约定
- 构建带循环的图（节点指回自身或前序节点）
- 掌握 `recursion_limit` 防止无限循环
- 理解条件分支 + 循环如何组合出 Agent 的「思考-行动」模式

## 0. 环境准备

In [ ]:
from importlib.metadata import version
print(f"langgraph 版本: {version('langgraph')}")

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="qwen-plus",
    openai_api_base=os.environ["Qwen_API_BASE"],
    openai_api_key=os.environ["Qwen_API_KEY"],
)

## 1. 回顾：线性图的局限

前两章的图都是线性的：`START → A → B → END`，数据只能从左到右。但现实场景经常需要：

- **分支**：根据条件走不同路径（如：情感是正面走路径 A，负面走路径 B）
- **循环**：重复执行直到满足条件（如：Agent 反复调用工具直到得出答案）

这就需要 `add_conditional_edges`。

## 2. 条件分支：add_conditional_edges

先看一个例子：根据用户输入的语言，走不同的翻译路径。

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    text: str
    language: str
    result: str

# 节点：检测语言
def detect_language(state: State) -> dict:
    response = llm.invoke(
        f"判断以下文本的语言，只回答 'chinese' 或 'english'，不要多余内容：\n{state['text']}"
    )
    return {"language": response.content.strip().lower()}

# 节点：中文 → 英文
def translate_to_english(state: State) -> dict:
    response = llm.invoke(f"翻译成英文，只输出翻译结果：\n{state['text']}")
    return {"result": response.content}

# 节点：英文 → 中文
def translate_to_chinese(state: State) -> dict:
    response = llm.invoke(f"翻译成中文，只输出翻译结果：\n{state['text']}")
    return {"result": response.content}

# 路由函数：根据 language 字段决定下一步
def route_by_language(state: State) -> Literal["to_english", "to_chinese"]:
    if "chinese" in state["language"]:
        return "to_english"
    return "to_chinese"

# 构建图
graph = StateGraph(State)
graph.add_node("detect", detect_language)
graph.add_node("to_english", translate_to_english)
graph.add_node("to_chinese", translate_to_chinese)

graph.add_edge(START, "detect")
graph.add_conditional_edges("detect", route_by_language)  # 条件分支
graph.add_edge("to_english", END)
graph.add_edge("to_chinese", END)

app = graph.compile()

In [ ]:
# 测试：输入中文
result = app.invoke({"text": "今天天气真好", "language": "", "result": ""})
print(f"检测语言: {result['language']}")
print(f"翻译结果: {result['result']}")

In [ ]:
# 测试：输入英文
result = app.invoke({"text": "The weather is great today", "language": "", "result": ""})
print(f"检测语言: {result['language']}")
print(f"翻译结果: {result['result']}")

In [ ]:
# 查看图结构
from IPython.display import Image
Image(app.get_graph().draw_mermaid_png())

### 刚才发生了什么？

`add_conditional_edges` 接收三个参数：

```python
graph.add_conditional_edges(
    source,           # 从哪个节点出发
    path,             # 路由函数：接收 state，返回目标节点名
    path_map=None,    # 可选：返回值到节点名的映射
)
```

| 参数 | 说明 |
|------|------|
| `source` | 条件分支的起点节点 |
| `path` | 路由函数，接收当前 state，返回目标节点名（字符串） |
| `path_map` | 可选。如果路由函数返回值和节点名不一致，用 dict 映射 |

路由函数的返回值直接就是目标节点名。用 `Literal` 类型标注可以让 IDE 和 LangGraph 知道所有可能的路径。

## 3. path_map：返回值映射

有时路由函数的返回值不想和节点名绑定（比如返回 `"yes"` / `"no"` 比返回节点名更语义化）。用 `path_map` 做映射：

In [ ]:
class ReviewState(TypedDict):
    text: str
    is_positive: str
    response: str

def analyze_sentiment(state: ReviewState) -> dict:
    response = llm.invoke(
        f"判断以下评论的情感，只回答 'positive' 或 'negative'：\n{state['text']}"
    )
    return {"is_positive": response.content.strip().lower()}

def handle_positive(state: ReviewState) -> dict:
    return {"response": "感谢您的好评！"}

def handle_negative(state: ReviewState) -> dict:
    response = llm.invoke(
        f"用户给出了差评：'{state['text']}'。请写一段安抚回复，不超过两句话。"
    )
    return {"response": response.content}

# 路由函数返回语义化的 yes/no
def route_sentiment(state: ReviewState) -> Literal["yes", "no"]:
    return "yes" if "positive" in state["is_positive"] else "no"

graph = StateGraph(ReviewState)
graph.add_node("analyze", analyze_sentiment)
graph.add_node("positive", handle_positive)
graph.add_node("negative", handle_negative)

graph.add_edge(START, "analyze")
graph.add_conditional_edges(
    "analyze",
    route_sentiment,
    {"yes": "positive", "no": "negative"},  # path_map：yes → positive 节点
)
graph.add_edge("positive", END)
graph.add_edge("negative", END)

app = graph.compile()

In [ ]:
# 测试正面评论
result = app.invoke({"text": "这个产品太棒了！", "is_positive": "", "response": ""})
print(f"情感: {result['is_positive']}")
print(f"回复: {result['response']}")

In [ ]:
# 测试负面评论
result = app.invoke({"text": "质量太差了，很失望", "is_positive": "", "response": ""})
print(f"情感: {result['is_positive']}")
print(f"回复: {result['response']}")

### 对比：有无 path_map

| 写法 | 路由函数返回 | 说明 |
|------|------------|------|
| 不用 path_map | 节点名本身（如 `"positive"`） | 简单直接，返回值 = 节点名 |
| 用 path_map | 语义化值（如 `"yes"`） | 路由逻辑和节点命名解耦 |

简单场景不用 path_map 即可，复杂场景用 path_map 更清晰。

## 4. 路由到 END

路由函数也可以直接返回 `"__end__"` 来终止图的执行。`END` 常量的值就是 `"__end__"`。

In [ ]:
from langgraph.graph import END

# END 的真实值
print(f"END = '{END}'")  # __end__

In [ ]:
from typing import Annotated
from operator import add

class GateState(TypedDict):
    query: str
    steps: Annotated[list[str], add]
    answer: str

def check_query(state: GateState) -> dict:
    return {"steps": ["checked query"]}

def process(state: GateState) -> dict:
    response = llm.invoke(state["query"])
    return {"answer": response.content, "steps": ["processed"]}

# 路由：空查询直接结束，非空才处理
def gate(state: GateState) -> Literal["process", "__end__"]:
    if not state["query"].strip():
        return "__end__"  # 直接返回 END
    return "process"

graph = StateGraph(GateState)
graph.add_node("check", check_query)
graph.add_node("process", process)
graph.add_edge(START, "check")
graph.add_conditional_edges("check", gate)
graph.add_edge("process", END)

app = graph.compile()

# 空查询：check 之后直接结束
result = app.invoke({"query": "", "steps": [], "answer": ""})
print(f"空查询 - steps: {result['steps']}, answer: '{result['answer']}'")

# 非空查询：check → process → END
result = app.invoke({"query": "1+1=?", "steps": [], "answer": ""})
print(f"正常查询 - steps: {result['steps']}, answer: '{result['answer'][:50]}'")

## 5. 循环：节点指回自身

条件边的目标可以是**任何已添加的节点**，包括自身或前序节点。这就形成了循环。

下面构建一个「自我改进」的例子：让模型写一段代码，然后自己检查，不满意就重写，直到满意为止。

In [ ]:
class RefineState(TypedDict):
    task: str
    draft: str
    feedback: str
    is_good: bool
    iterations: Annotated[int, add]

def write_draft(state: RefineState) -> dict:
    if state["draft"]:
        # 有反馈，根据反馈修改
        prompt = f"任务：{state['task']}\n上一版：{state['draft']}\n反馈：{state['feedback']}\n请修改，只输出代码。"
    else:
        prompt = f"任务：{state['task']}\n请写出代码，只输出代码。"
    response = llm.invoke(prompt)
    return {"draft": response.content, "iterations": 1}

def review_draft(state: RefineState) -> dict:
    prompt = f"""审查以下代码，任务要求是：{state['task']}

代码：
{state['draft']}

如果代码正确且符合要求，回答 'PASS'。否则指出问题（一句话）。"""
    response = llm.invoke(prompt)
    content = response.content.strip()
    is_good = "pass" in content.lower()
    return {"feedback": content, "is_good": is_good}

def should_continue(state: RefineState) -> Literal["write", "__end__"]:
    if state["is_good"]:
        return "__end__"
    return "write"  # 回到 write 节点，形成循环

graph = StateGraph(RefineState)
graph.add_node("write", write_draft)
graph.add_node("review", review_draft)
graph.add_edge(START, "write")
graph.add_edge("write", "review")
graph.add_conditional_edges("review", should_continue)  # 循环的关键

app = graph.compile()
from IPython.display import Image
Image(app.get_graph().draw_mermaid_png())

In [ ]:
# 运行并观察每一步
for step in app.stream({
    "task": "写一个 Python 函数 is_palindrome(s)，判断字符串是否为回文",
    "draft": "",
    "feedback": "",
    "is_good": False,
    "iterations": 0,
}):
    for node_name, update in step.items():
        print(f"=== 节点 [{node_name}] ===")
        if "draft" in update:
            print(f"  代码: {update['draft'][:100]}...")
        if "feedback" in update:
            print(f"  反馈: {update['feedback'][:100]}")
        if "is_good" in update:
            print(f"  通过: {update['is_good']}")
        if "iterations" in update:
            print(f"  迭代: +{update['iterations']}")
        print()

### 刚才发生了什么？

1. `write` 节点生成代码（首次创作或根据反馈修改）
2. `review` 节点审查代码，判断是否通过
3. `should_continue` 路由函数检查 `is_good`：
   - `True` → 结束
   - `False` → 回到 `write` 节点（**循环**）

这就是「循环」的实现方式：条件边指向一个已经执行过的节点。

## 6. recursion_limit：防止无限循环

循环如果退出条件写错了，会无限执行。LangGraph 默认有 `recursion_limit=25` 的保护。可以在 `invoke` / `stream` 时覆盖：

In [ ]:
# 构造一个永远不会结束的循环来演示
class LoopState(TypedDict):
    count: Annotated[int, add]

def increment(state: LoopState) -> dict:
    return {"count": 1}

def always_continue(state: LoopState) -> Literal["increment", "__end__"]:
    return "increment"  # 永远循环

graph = StateGraph(LoopState)
graph.add_node("increment", increment)
graph.add_edge(START, "increment")
graph.add_conditional_edges("increment", always_continue)

app = graph.compile()

# 设置较小的 recursion_limit
try:
    app.invoke({"count": 0}, {"recursion_limit": 5})
except Exception as e:
    print(f"触发限制: {type(e).__name__}")
    print(f"消息: {str(e)[:100]}")

### recursion_limit 参数

| 设置方式 | 示例 |
|---------|------|
| invoke 时指定 | `app.invoke(input, {"recursion_limit": 10})` |
| stream 时指定 | `app.stream(input, {"recursion_limit": 10})` |
| 默认值 | 25 |

**注意**：`recursion_limit` 是总步数限制（每个节点执行一次算一步），不是循环次数。两个节点的循环，limit=10 意味着最多循环 5 次。

## 7. 实战：带条件分支和循环的完整例子

把前面学的组合起来：一个简易的数学辅导助手。

1. 用户提问
2. 模型判断是否为数学问题（条件分支）
3. 如果是数学问题 → 解答 → 自检答案是否正确 → 不正确就重新解答（循环）
4. 如果不是数学问题 → 直接拒绝

In [ ]:
class TutorState(TypedDict):
    question: str
    is_math: bool
    answer: str
    is_correct: bool
    attempts: Annotated[int, add]

def classify(state: TutorState) -> dict:
    response = llm.invoke(
        f"判断这是否为数学问题，只回答 'yes' 或 'no'：\n{state['question']}"
    )
    return {"is_math": "yes" in response.content.strip().lower()}

def solve(state: TutorState) -> dict:
    prompt = f"请解答以下数学问题，给出计算过程和最终答案：\n{state['question']}"
    if state["answer"]:
        prompt += f"\n\n上次的答案有误：{state['answer']}\n请重新解答。"
    response = llm.invoke(prompt)
    return {"answer": response.content, "attempts": 1}

def verify(state: TutorState) -> dict:
    response = llm.invoke(
        f"检验以下数学解答是否正确，只回答 'correct' 或 'wrong'：\n"
        f"问题：{state['question']}\n答案：{state['answer']}"
    )
    return {"is_correct": "correct" in response.content.strip().lower()}

def reject(state: TutorState) -> dict:
    return {"answer": "抱歉，我只能回答数学问题。"}

# 路由 1：是否为数学问题
def route_math(state: TutorState) -> Literal["solve", "reject"]:
    return "solve" if state["is_math"] else "reject"

# 路由 2：答案是否正确（最多 3 次）
def route_verify(state: TutorState) -> Literal["solve", "__end__"]:
    if state["is_correct"] or state["attempts"] >= 3:
        return "__end__"
    return "solve"

graph = StateGraph(TutorState)
graph.add_node("classify", classify)
graph.add_node("solve", solve)
graph.add_node("verify", verify)
graph.add_node("reject", reject)

graph.add_edge(START, "classify")
graph.add_conditional_edges("classify", route_math)      # 分支
graph.add_edge("solve", "verify")
graph.add_conditional_edges("verify", route_verify)       # 循环
graph.add_edge("reject", END)

app = graph.compile()
from IPython.display import Image
Image(app.get_graph().draw_mermaid_png())

In [ ]:
# 测试数学问题
for step in app.stream({
    "question": "计算 17 × 23 的结果",
    "is_math": False, "answer": "", "is_correct": False, "attempts": 0,
}):
    for node, update in step.items():
        print(f"[{node}]", {k: str(v)[:60] for k, v in update.items()})

In [ ]:
# 测试非数学问题
result = app.invoke({
    "question": "推荐一部好看的电影",
    "is_math": False, "answer": "", "is_correct": False, "attempts": 0,
})
print(result["answer"])

### 这个例子展示了什么？

一个图中同时有**条件分支**和**循环**：

- `classify` → 分支：数学走 `solve`，非数学走 `reject`
- `solve` → `verify` → 循环：不正确就回到 `solve`，正确或超过 3 次则结束

这正是 Agent 的基本模式——下一章我们会用 LangGraph 的内置工具支持来实现真正的 Agent。

## 总结

本章学习了：
- ✅ `add_conditional_edges` 实现动态路由
- ✅ 路由函数的写法：接收 state，返回节点名或 `"__end__"`
- ✅ `path_map` 参数解耦返回值与节点名
- ✅ 条件边指向已执行节点实现循环
- ✅ `recursion_limit` 防止无限循环
- ✅ 分支 + 循环组合的完整应用

## 下一章预告

**第四章：工具调用 Agent（Tool Node、ReAct 模式）** —— 学习如何让模型调用外部工具（搜索、计算、API），并用 LangGraph 的 `ToolNode` 和 `create_react_agent` 构建真正的 Agent。